# DeepSurv (PyCox) - Training Pipeline

Questo notebook definisce l'addestramento di un modello di Deep Learning per l'analisi di sopravvivenza (DeepSurv) utilizzando la libreria PyCox. A differenza dei modelli basati su alberi, le reti neurali richiedono una preparazione dei dati profondamente diversa e accorgimenti specifici per evitare l'overfitting.

---

### Fase 1: Caricamento e Split
Il dataset viene caricato nella sua forma grezza. I valori sentinella `-1` vengono convertiti in `NaN` e i missing indicators vengono creati immediatamente dopo, in modo che siano disponibili come feature per entrambi i set. Lo split train/test (75%/25%) viene eseguito **immediatamente**, prima di qualsiasi operazione che calcoli statistiche sul dataset. In questo modo nessuna decisione successiva — feature selection, imputazione, scaling — può essere influenzata dai dati di test.

### Fase 2: Feature Selection Post-Split
Il Variance Threshold (rimozione delle feature binarie troppo sparse) viene calcolato **esclusivamente sui dati di training**. La lista delle colonne da rimuovere viene poi applicata anche al test set. Eseguire questa operazione prima dello split costituirebbe data leakage: la somma dei positivi di una feature dipenderebbe anche dai campioni di test, potendo modificare la decisione di tenerla o scartarla.

### Fase 3: Imputazione e Scaling
Poiché le reti neurali e gli scaler non tollerano valori mancanti, applichiamo l'algoritmo MICE (`IterativeImputer`), addestrato (`fit`) **esclusivamente sul Train**, per risolvere i `NaN`. Analogamente, lo `StandardScaler` viene addestrato solo sul train imputato e applicato con `transform` agli altri set. Imputer e scaler vengono salvati in `.pkl` per essere riutilizzati in evaluation.

### Fase 4: Architettura della Rete Neurale (DeepSurv)
Costruiamo un Multi-Layer Perceptron (MLPVanilla) configurato per limitare la complessità ed evitare di memorizzare il dataset (overfitting):
* **Strati Nascosti:** Due layer densi molto piccoli (32 e 16 nodi) proporzionati alle dimensioni ridotte del dataset.
* **Batch Normalization:** Attivata per stabilizzare e accelerare la convergenza dei gradienti durante l'addestramento.
* **Dropout (0.4):** Una tecnica di regolarizzazione aggressiva che "spegne" casualmente il 40% dei neuroni ad ogni iterazione, costringendo la rete ad apprendere pattern robusti e distribuiti.

### Fase 5: Addestramento ed Early Stopping
L'addestramento utilizza l'ottimizzatore Adam con un learning rate di 0.01 per un massimo di 200 epoche. Per garantire la massima generalizzazione, introduciamo una callback di **Early Stopping** (patience=15): se l'errore sul Validation Set smette di diminuire per 15 epoche consecutive, l'addestramento si interrompe automaticamente, prevenendo l'overfitting.

### Fase 6: Salvataggio Modello
La rete neurale viene salvata nel formato nativo di PyTorch (`.pt`), mentre lo scaler, l'imputer e la lista delle feature finali vengono salvati separatamente. Tutti gli artefatti verranno ricaricati in fase di evaluation.

In [23]:
import os, warnings
import pandas as pd
import numpy as np
import joblib
import torchtuples as tt
from pycox.models import CoxPH
from pycox.evaluation import EvalSurv
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer

warnings.filterwarnings("ignore")

# ==============================================================================
# CONFIGURAZIONE E PATHS
# ==============================================================================
DATA_PATH     = "../data/dataset.csv"
MODEL_DIR     = "../models"
MODEL_PATH    = os.path.join(MODEL_DIR, "deepsurv_model.pt")
SCALER_PATH   = os.path.join(MODEL_DIR, "deepsurv_scaler.pkl")
IMPUTER_PATH  = os.path.join(MODEL_DIR, "deepsurv_imputer.pkl")
FEATURES_PATH = os.path.join(MODEL_DIR, "deepsurv_features.pkl")

os.makedirs(MODEL_DIR, exist_ok=True)

TIME_COL     = "follow_ups.days_to_follow_up"
EVENT_COL    = "demographic.vital_status_DEAD"
RANDOM_STATE = 26
TEST_SIZE    = 0.25
VAL_SIZE     = 0.15

NET_LAYERS   = [32, 32, 16]
NET_DROPOUT  = 0.4
NET_BATCHNORM = True

# ==============================================================================
# FASE 1: CARICAMENTO E SPLIT
# ==============================================================================
print("=" * 62)
print("  TRAINING PIPELINE - DEEPSURV (PyCox)")
print("=" * 62)

df_clean = pd.read_csv(DATA_PATH).replace(-1, np.nan)

X_raw = df_clean.drop(columns=[TIME_COL, EVENT_COL])

cols_with_many_nan = ['exposures.alcohol_days_per_week',
                      'exposures.alcohol_drinks_per_day',
                      'exposures.pack_years_smoked']

for col in cols_with_many_nan:
    if col in X_raw.columns:
        X_raw[f"{col}_is_missing"] = X_raw[col].isna().astype(float)

X_train_full, X_test, df_y_train_full, df_y_test = train_test_split(
    X_raw, df_clean[[TIME_COL, EVENT_COL]],
    test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=df_clean[EVENT_COL]
)

# ==============================================================================
# FASE 2: FEATURE SELECTION (Frequenze) E VALIDATION SPLIT
# ==============================================================================
min_positives = 10
binary_cols   = [c for c in X_train_full.columns if X_train_full[c].nunique() == 2]
cols_to_drop  = [c for c in binary_cols if X_train_full[c].sum() < min_positives]

X_train_full = X_train_full.drop(columns=cols_to_drop)
X_test       = X_test.drop(columns=cols_to_drop, errors='ignore')
print(f"  [+] Rimosse {len(cols_to_drop)} feature troppo sparse.")

X_train, X_val, df_y_train, df_y_val = train_test_split(
    X_train_full, df_y_train_full,
    test_size=VAL_SIZE, random_state=RANDOM_STATE, stratify=df_y_train_full[EVENT_COL]
)

joblib.dump(X_train.columns.tolist(), FEATURES_PATH)

# ==============================================================================
# FASE 3: IMPUTAZIONE E SCALING COMPLETO (Senza Leakage)
# ==============================================================================
imputer = IterativeImputer(random_state=RANDOM_STATE, max_iter=10)
X_train_imp = pd.DataFrame(imputer.fit_transform(X_train), columns=X_train.columns)
X_val_imp   = pd.DataFrame(imputer.transform(X_val),       columns=X_val.columns)
X_test_imp  = pd.DataFrame(imputer.transform(X_test),      columns=X_test.columns)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_imp).astype('float32')
X_val_scaled   = scaler.transform(X_val_imp).astype('float32')
X_test_scaled  = scaler.transform(X_test_imp).astype('float32')

joblib.dump(imputer, IMPUTER_PATH)
joblib.dump(scaler,  SCALER_PATH)

get_target = lambda df: (df[TIME_COL].values.astype('float32'), df[EVENT_COL].values.astype('float32'))
y_train = get_target(df_y_train)
y_val   = get_target(df_y_val)
y_test  = get_target(df_y_test)

# ==============================================================================
# FASE 4: ARCHITETTURA RETE E TRAINING
# ==============================================================================
in_features = X_train_scaled.shape[1]

net = tt.practical.MLPVanilla(
    in_features, NET_LAYERS, 1,
    batch_norm=NET_BATCHNORM,
    dropout=NET_DROPOUT
)

optimizer = tt.optim.Adam(lr=0.001, weight_decay=0.01)
model = CoxPH(net, optimizer)

callbacks = [tt.callbacks.EarlyStopping(patience=25)]

print(f"  [+] Avvio addestramento su {in_features} feature...")
log = model.fit(X_train_scaled, y_train, batch_size=64, epochs=500,
                callbacks=callbacks, val_data=(X_val_scaled, y_val), verbose=False)

model.save_net(MODEL_PATH)

# ==============================================================================
# FASE 5: VALUTAZIONE FINALE C-INDEX
# ==============================================================================
_ = model.compute_baseline_hazards()

def evaluate_c_index(model_obj, x_data, y_data):
    durations, events = y_data
    surv_df = model_obj.predict_surv_df(x_data)
    ev = EvalSurv(surv_df, durations, events, censor_surv='km')
    return ev.concordance_td()

c_index_train = evaluate_c_index(model, X_train_scaled, y_train)
c_index_val   = evaluate_c_index(model, X_val_scaled,   y_val)
c_index_test  = evaluate_c_index(model, X_test_scaled,  y_test)

print("=" * 62)
print("  RISULTATI DEEPSURV")
print("=" * 62)
print(f"  Epoche ottimali : {len(log.to_pandas())}")
print(f"  C-index TRAIN   : {c_index_train:.4f}")
print(f"  C-index VAL     : {c_index_val:.4f}")
print(f"  C-index TEST    : {c_index_test:.4f}")
print("=" * 62)

  TRAINING PIPELINE - DEEPSURV (PyCox)
  [+] Rimosse 1 feature troppo sparse.
  [+] Avvio addestramento su 35 feature...
  RISULTATI DEEPSURV
  Epoche ottimali : 119
  C-index TRAIN   : 0.7903
  C-index VAL     : 0.6123
  C-index TEST    : 0.7239


# DeepSurv (PyCox) - Evaluation & Explainable AI

Questo notebook valuta le performance della rete neurale DeepSurv e ne esplora i meccanismi decisionali. Date le peculiarità architetturali di PyCox, la pipeline di valutazione e spiegabilità richiede alcuni passaggi di adattamento rispetto ai modelli standard.

---

### Fase 1: Ricostruzione Ambiente, Imputazione, Scaling e Baseline Hazards
Per valutare la rete, replichiamo rigorosamente l'ambiente di training:
* Il dataset viene caricato grezzo e i missing indicators vengono ricreati identicamente al training.
* Lo split viene eseguito immediatamente con lo stesso `random_state`, **prima** di qualsiasi trasformazione, garantendo che il test set sia identico a quello del training.
* La feature selection (rimozione delle colonne sparse) viene rieseguita calcolando `cols_to_drop` **solo sul train**, esattamente come in fase di addestramento, e poi applicata ad entrambi i set.
* L'imputer MICE e lo `StandardScaler` salvati vengono ricaricati e applicati con solo `transform` — mai `fit` — sui dati di evaluation.
* **Baseline Hazards:** DeepSurv in PyCox calcola un rischio relativo. Per ottenere funzioni di sopravvivenza assolute ($S(t)$) e calcolare l'Integrated Brier Score, dobbiamo prima calcolare le "Baseline Hazards" passando l'intero set di training alla rete.

### Fase 2: Metriche Globali di Sopravvivenza
Le probabilità predette vengono appiattite e passate alle metriche standard:
* **C-index:** Misura l'accuratezza del ranking di rischio predetto dalla rete.
* **Integrated Brier Score (IBS):** Il dataframe di sopravvivenza predetto da PyCox viene interpolato per allinearlo ai step temporali di valutazione richiesti da `sksurv`, ottenendo così una misura di calibrazione dell'errore nel tempo.
* **Time-dependent AUC:** Valuta l'AUC a step temporali specifici (1, 3 e 5 anni).

### Fase 3: Stratificazione del Rischio (Validazione Clinica)
I pazienti del test set vengono separati in coorti ad "Alto Rischio" e "Basso Rischio" basandosi sulla mediana del rischio relativo (output grezzo della rete).
Successivamente, si mediano le curve di sopravvivenza (derivate dal dataframe di PyCox) per verificare visivamente l'efficacia del modello nel differenziare le prognosi.

### Fase 4: Explainable AI Globale (Permutation Feature Importance)
Le reti neurali sono notoriamente delle "black box". Per ispezionarle globalmente, avvolgiamo il modello DeepSurv in una classe wrapper (`PyCoxWrapper`) compatibile con scikit-learn. Questo permette di eseguire la Permutation Feature Importance, alterando una variabile scalata alla volta per misurare la conseguente diminuzione del C-index, rivelando così quali feature dominano l'apprendimento globale.

### Fase 5: Explainable AI Locale (SHAP) con Dati Imputati e Comprensibili
Per analizzare le decisioni sui singoli pazienti usiamo SHAP con un accorgimento tecnico cruciale:
Creiamo un wrapper personalizzato (`predict_risk_shap`) che accetta in input i dati **imputati ma non scalati**, applica lo scaler internamente e restituisce il rischio.
Questo permette al KernelExplainer di SHAP di campionare e restituire grafici (Summary Plot e Waterfall Plot) che mostrano i **valori originali e comprensibili delle feature** (es. "Età = 65"), anziché valori astratti standardizzati (es. "Età = 1.24"), rendendo l'interpretazione clinicamente utile e intellegibile.

In [27]:
import os, warnings
import pandas as pd
import numpy as np
import joblib
import torchtuples as tt
import shap
import matplotlib.pyplot as plt
from pycox.models import CoxPH
from sksurv.util import Surv
from sksurv.metrics import (concordance_index_censored,
                            integrated_brier_score,
                            cumulative_dynamic_auc)
from sklearn.model_selection import train_test_split
from sklearn.inspection import permutation_importance

warnings.filterwarnings("ignore")

# ==============================================================================
# CONFIGURAZIONE E PATHS
# ==============================================================================
RESULTS_DIR   = "results"
DATA_PATH     = "../data/dataset.csv"
MODEL_DIR     = "../models"
MODEL_PATH    = os.path.join(MODEL_DIR, "deepsurv_model.pt")
SCALER_PATH   = os.path.join(MODEL_DIR, "deepsurv_scaler.pkl")
IMPUTER_PATH  = os.path.join(MODEL_DIR, "deepsurv_imputer.pkl")
FEATURES_PATH = os.path.join(MODEL_DIR, "deepsurv_features.pkl")

os.makedirs(RESULTS_DIR, exist_ok=True)

for _f in [MODEL_PATH, SCALER_PATH, IMPUTER_PATH, FEATURES_PATH]:
    if not os.path.exists(_f):
        raise FileNotFoundError(f"File {_f} non trovato. Esegui prima il training.")

TIME_COL     = "follow_ups.days_to_follow_up"
EVENT_COL    = "demographic.vital_status_DEAD"
RANDOM_STATE = 26
TEST_SIZE    = 0.25
VAL_SIZE     = 0.15
N_RUNS       = 5

NET_LAYERS    = [64, 32, 16]
NET_DROPOUT   = 0.4
NET_BATCHNORM = True

# ==============================================================================
# FASE 1: RICOSTRUZIONE AMBIENTE E PREPROCESSING (UNA SOLA VOLTA)
# ==============================================================================
print("=" * 62)
print("  EVALUATION & XAI - DEEPSURV  (media su 5 run)")
print("=" * 62)

df_clean = pd.read_csv(DATA_PATH).replace(-1, np.nan)

X_raw = df_clean.drop(columns=[TIME_COL, EVENT_COL])

cols_with_many_nan = ['exposures.alcohol_days_per_week',
                      'exposures.alcohol_drinks_per_day',
                      'exposures.pack_years_smoked']

for col in cols_with_many_nan:
    if col in X_raw.columns:
        X_raw[f"{col}_is_missing"] = X_raw[col].isna().astype(float)

X_train_full, X_test, df_y_train_full, df_y_test = train_test_split(
    X_raw, df_clean[[TIME_COL, EVENT_COL]],
    test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=df_clean[EVENT_COL]
)

min_positives = 10
binary_cols   = [c for c in X_train_full.columns if X_train_full[c].nunique() == 2]
cols_to_drop  = [c for c in binary_cols if X_train_full[c].sum() < min_positives]

X_train_full = X_train_full.drop(columns=cols_to_drop)
X_test       = X_test.drop(columns=cols_to_drop, errors='ignore')

feature_cols_final = joblib.load(FEATURES_PATH)

for col in feature_cols_final:
    if col not in X_train_full.columns:
        X_train_full[col] = 0
    if col not in X_test.columns:
        X_test[col] = 0

X_train_full = X_train_full[feature_cols_final]
X_test       = X_test[feature_cols_final]

y_train_full_sksurv = Surv.from_arrays(
    event=df_y_train_full[EVENT_COL].astype(bool),
    time=df_y_train_full[TIME_COL]
)
y_test_sksurv = Surv.from_arrays(
    event=df_y_test[EVENT_COL].astype(bool),
    time=df_y_test[TIME_COL]
)

imputer = joblib.load(IMPUTER_PATH)
scaler  = joblib.load(SCALER_PATH)

X_train_full_imp = pd.DataFrame(imputer.transform(X_train_full), columns=X_train_full.columns)
X_test_imp       = pd.DataFrame(imputer.transform(X_test),       columns=X_test.columns)

X_train_full_scaled = scaler.transform(X_train_full_imp).astype('float32')
X_test_scaled       = scaler.transform(X_test_imp).astype('float32')

y_train_full_pycox = (
    df_y_train_full[TIME_COL].values.astype('float32'),
    df_y_train_full[EVENT_COL].values.astype('float32')
)

in_features = X_train_full_scaled.shape[1]
print(f"  [+] Ambiente ricostruito: {in_features} feature attive.")

lb        = np.percentile(df_y_test[TIME_COL].values, 10)
ub        = np.percentile(df_y_test[TIME_COL].values, 90)
ibs_times = np.linspace(lb, ub, 20)

time_max_safe  = min(df_y_train_full[TIME_COL].max(), df_y_test[TIME_COL].max())
horizon_labels = {1: 365, 3: 1095, 5: 1825}
valid_horizons = {lbl: days for lbl, days in horizon_labels.items() if days < time_max_safe}
skipped        = [lbl for lbl, days in horizon_labels.items() if days >= time_max_safe]

# ==============================================================================
# FASE 2: LOOP SU N_RUNS — TRAINING E METRICHE
# ==============================================================================
print(f"\n  [+] Avvio {N_RUNS} run indipendenti...")
print("=" * 62)

runs_c_index = []
runs_ibs     = []
runs_auc     = {lbl: [] for lbl in valid_horizons}
runs_models  = []
runs_risk    = []
runs_surv    = []

for run in range(N_RUNS):
    run_seed = RANDOM_STATE + run

    X_tr, X_vl, df_ytr, df_yvl = train_test_split(
        X_train_full, df_y_train_full,
        test_size=VAL_SIZE, random_state=run_seed,
        stratify=df_y_train_full[EVENT_COL]
    )

    X_tr_imp = pd.DataFrame(imputer.transform(X_tr), columns=X_tr.columns)
    X_vl_imp = pd.DataFrame(imputer.transform(X_vl), columns=X_vl.columns)

    X_tr_scaled = scaler.transform(X_tr_imp).astype('float32')
    X_vl_scaled = scaler.transform(X_vl_imp).astype('float32')

    y_tr = (df_ytr[TIME_COL].values.astype('float32'), df_ytr[EVENT_COL].values.astype('float32'))
    y_vl = (df_yvl[TIME_COL].values.astype('float32'), df_yvl[EVENT_COL].values.astype('float32'))

    import torch; torch.manual_seed(run_seed); np.random.seed(run_seed)
    net   = tt.practical.MLPVanilla(in_features, NET_LAYERS, 1, NET_BATCHNORM, NET_DROPOUT)
    model = CoxPH(net, tt.optim.Adam(lr=0.001, weight_decay=0.01))

    model.fit(X_tr_scaled, y_tr, batch_size=64, epochs=500,
              callbacks=[tt.callbacks.EarlyStopping(patience=25)],
              val_data=(X_vl_scaled, y_vl), verbose=False)

    _ = model.compute_baseline_hazards(X_train_full_scaled, y_train_full_pycox)

    risk_test = model.predict(X_test_scaled).flatten()
    surv_df   = model.predict_surv_df(X_test_scaled)

    c_idx = concordance_index_censored(
        y_test_sksurv['event'], y_test_sksurv['time'], risk_test
    )[0]

    preds_interp = np.array([
        np.interp(ibs_times, surv_df.index.values, surv_df.iloc[:, i].values)
        for i in range(surv_df.shape[1])
    ])
    ibs_val = integrated_brier_score(
        y_train_full_sksurv, y_test_sksurv, preds_interp, ibs_times
    )

    auc_vals = {}
    if valid_horizons:
        auc_times  = np.array(list(valid_horizons.values()), dtype=float)
        auc_result, _ = cumulative_dynamic_auc(
            y_train_full_sksurv, y_test_sksurv, risk_test, auc_times
        )
        for lbl, av in zip(valid_horizons.keys(), auc_result):
            auc_vals[lbl] = av

    runs_c_index.append(c_idx)
    runs_ibs.append(ibs_val)
    for lbl in valid_horizons:
        runs_auc[lbl].append(auc_vals[lbl])
    runs_models.append(model)
    runs_risk.append(risk_test)
    runs_surv.append(surv_df)

    print(f"  Run {run + 1}/{N_RUNS} — C-index: {c_idx:.4f} | IBS: {ibs_val:.4f}")

# ==============================================================================
# FASE 3: AGGREGAZIONE RISULTATI
# ==============================================================================
print("\n" + "=" * 62)
print("  FASE 3: METRICHE AGGREGATE (media ± std)")
print("=" * 62)

mean_c   = np.mean(runs_c_index)
std_c    = np.std(runs_c_index)
mean_ibs = np.mean(runs_ibs)
std_ibs  = np.std(runs_ibs)

print(f"  C-index : {mean_c:.4f} ± {std_c:.4f}")
print(f"  IBS     : {mean_ibs:.4f} ± {std_ibs:.4f}")

if skipped:
    print(f"  [!] Orizzonti esclusi (oltre il follow-up massimo): {skipped} anni")

mean_auc_overall = []
for lbl, days in valid_horizons.items():
    m = np.mean(runs_auc[lbl])
    s = np.std(runs_auc[lbl])
    mean_auc_overall.append(m)
    print(f"  AUC a {lbl} anno/i ({days} giorni) : {m:.4f} ± {s:.4f}")

if mean_auc_overall:
    print(f"  AUC media orizzonti validi         : {np.mean(mean_auc_overall):.4f}")

best_run_idx = int(np.argmin(np.abs(np.array(runs_c_index) - mean_c)))
model     = runs_models[best_run_idx]
risk_test = runs_risk[best_run_idx]
surv_df   = runs_surv[best_run_idx]
print(f"\n  [+] Modello rappresentativo: run {best_run_idx + 1} (C-index: {runs_c_index[best_run_idx]:.4f})")

# ==============================================================================
# FASE 4: STRATIFICAZIONE DEL RISCHIO
# ==============================================================================
print("\n" + "=" * 62)
print("  FASE 4: STRATIFICAZIONE CLINICA")
print("=" * 62)

median_risk = np.median(risk_test)
mask_high   = risk_test > median_risk
mask_low    = risk_test <= median_risk

plt.figure(figsize=(10, 6))
plt.step(surv_df.index, surv_df.loc[:, mask_low].mean(axis=1),  where="post", label="Basso Rischio", color="#2ca02c")
plt.step(surv_df.index, surv_df.loc[:, mask_high].mean(axis=1), where="post", label="Alto Rischio",  color="#d62728")
plt.title("Sopravvivenza Media: Alto vs Basso Rischio (DeepSurv)")
plt.ylabel("S(t)")
plt.xlabel("Giorni")
plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, "deepsurv_survival_curves.png"), dpi=150)
plt.close()
print(f"  [+] Grafico curve salvato in: {RESULTS_DIR}")

# ==============================================================================
# FASE 5: XAI GLOBALE (PERMUTATION FEATURE IMPORTANCE)
# ==============================================================================
print("\n" + "=" * 62)
print("  FASE 5: PERMUTATION FEATURE IMPORTANCE")
print("=" * 62)
print("  Calcolo in corso (10 ripetizioni)...")

def deepsurv_scorer(est, X_in, y_in):
    risk = est.predict(X_in.astype('float32')).flatten()
    return concordance_index_censored(y_in['event'], y_in['time'], risk)[0]

perm = permutation_importance(
    model, X_test_scaled, y_test_sksurv,
    scoring=deepsurv_scorer,
    n_repeats=10,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

imp_mean = np.maximum(perm.importances_mean, 0)
imp_pct  = 100.0 * imp_mean / (imp_mean.sum() + 1e-12)

df_imp = pd.DataFrame({
    "Feature":        X_test.columns,
    "Importanza (%)": imp_pct,
    "Std":            perm.importances_std,
}).sort_values("Importanza (%)", ascending=False)

fig2, ax2 = plt.subplots(figsize=(9, 7))
ax2.barh(df_imp["Feature"][::-1], df_imp["Importanza (%)"][::-1],
         color="#1f77b4", xerr=df_imp["Std"][::-1], capsize=3)
ax2.set_xlabel("Importanza relativa (%)", fontsize=11)
ax2.set_title("Permutation Feature Importance – DeepSurv", fontsize=12, fontweight="bold")
ax2.grid(axis="x", alpha=0.35, linestyle="--")
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, "deepsurv_permutation_importance.png"), dpi=150)
plt.close()
print(f"  [+] Grafico Permutation salvato in: {RESULTS_DIR}")

# ==============================================================================
# FASE 6: XAI LOCALE (SHAP)
# ==============================================================================
print("\n" + "=" * 62)
print("  FASE 6: XAI LOCALE (SHAP)")
print("=" * 62)

def predict_risk_shap(X_imp):
    X_scaled = scaler.transform(X_imp).astype('float32')
    return model.predict(X_scaled).flatten()

background = shap.sample(X_train_full_imp, 100, random_state=RANDOM_STATE)
explainer  = shap.Explainer(predict_risk_shap, background)

print("  Calcolo SHAP (BeeSwarm)...")
shap_values = explainer(X_test_imp)

plt.figure(figsize=(10, 8))
shap.plots.beeswarm(shap_values, max_display=15, show=False)
plt.title("SHAP Summary: Impatto sul Rischio (con Missing Indicators)", fontsize=12, fontweight="bold")
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, "deepsurv_shap_summary.png"), dpi=150, bbox_inches="tight")
plt.close()
print("  [+] SHAP Summary plot salvato")

idx_alto  = int(np.argmax(risk_test))
idx_basso = int(np.argmin(risk_test))

for label, idx in [("ALTO RISCHIO", idx_alto), ("BASSO RISCHIO", idx_basso)]:
    plt.figure(figsize=(14, 7))
    try:
        shap.plots.waterfall(shap_values[idx], max_display=13, show=False, matplotlib=True)
    except TypeError:
        shap.plots.waterfall(shap_values[idx], max_display=13, show=False)

    ax_wf = plt.gca()
    plt.title(f"SHAP Waterfall – DeepSurv – Paziente {label}", fontsize=12, fontweight="bold", pad=12)
    ax_wf.tick_params(axis="y", labelsize=9)
    for text in ax_wf.texts:
        if "f(x)" not in text.get_text() and "base value" not in text.get_text():
            text.set_fontsize(8)
    plt.subplots_adjust(left=0.42, right=0.95, top=0.90, bottom=0.10)
    ax_wf.grid(axis="x", alpha=0.3, linestyle="--")

    fname = f"deepsurv_shap_waterfall_{label.lower().replace(' ', '_')}.png"
    plt.savefig(os.path.join(RESULTS_DIR, fname), dpi=150, bbox_inches="tight")
    plt.close()

print("  [+] SHAP Waterfall plots salvati per pazienti ad alto e basso rischio")

# ==============================================================================
# RIEPILOGO FINALE
# ==============================================================================
print("\n" + "=" * 62)
print("  RIEPILOGO FINALE – DeepSurv")
print("=" * 62)
print(f"  Run eseguiti    : {N_RUNS}")
print(f"  C-index         : {mean_c:.4f} ± {std_c:.4f}")
print(f"  IBS             : {mean_ibs:.4f} ± {std_ibs:.4f}")
for lbl, days in valid_horizons.items():
    m = np.mean(runs_auc[lbl])
    s = np.std(runs_auc[lbl])
    print(f"  AUC {lbl}yr        : {m:.4f} ± {s:.4f}")
print(f"  Grafici salvati in: {RESULTS_DIR}")
print("=" * 62)

  EVALUATION & XAI - DEEPSURV  (media su 5 run)
  [+] Ambiente ricostruito: 35 feature attive.

  [+] Avvio 5 run indipendenti...
  Run 1/5 — C-index: 0.7226 | IBS: 0.1850
  Run 2/5 — C-index: 0.7157 | IBS: 0.1908
  Run 3/5 — C-index: 0.7114 | IBS: 0.1945
  Run 4/5 — C-index: 0.6825 | IBS: 0.2027
  Run 5/5 — C-index: 0.6959 | IBS: 0.2083

  FASE 3: METRICHE AGGREGATE (media ± std)
  C-index : 0.7056 ± 0.0145
  IBS     : 0.1963 ± 0.0083
  AUC a 1 anno/i (365 giorni) : 0.7477 ± 0.0238
  AUC a 3 anno/i (1095 giorni) : 0.7347 ± 0.0192
  AUC a 5 anno/i (1825 giorni) : 0.6221 ± 0.0349
  AUC media orizzonti validi         : 0.7015

  [+] Modello rappresentativo: run 3 (C-index: 0.7114)

  FASE 4: STRATIFICAZIONE CLINICA
  [+] Grafico curve salvato in: results

  FASE 5: PERMUTATION FEATURE IMPORTANCE
  Calcolo in corso (10 ripetizioni)...
  [+] Grafico Permutation salvato in: results

  FASE 6: XAI LOCALE (SHAP)
  Calcolo SHAP (BeeSwarm)...
  [+] SHAP Summary plot salvato
  [+] SHAP Waterfall